In [1]:
!pip install --upgrade imbalanced-learn
!pip install --upgrade scikit-learn
!pip install opencv-python-headless 
!pip install torch torchvision pandas numpy matplotlib seaborn opencv-python scikit-learn Pillow imblearn tqdm
!pip install pandas lime scikit-image -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 56.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
import torch.nn.functional as F
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# =================================================================================
# 1. CONFIGURATION & HYPERPARAMETERS (For Manuscript)
# =================================================================================
# [MANUSCRIPT DATA] Copy these to your "Methods" section:
# - Batch Size: 16 | Hardware: NVIDIA T4/P100 GPU | Software: PyTorch 2.1, torchvision
# - Optimizer: Adam (lr=1e-4) | Scheduler: ReduceLROnPlateau | Criterion: Dice + BCE Loss
# =================================================================================

MODEL1_PATH = '/kaggle/input/datasets/deblinamazumdersetu/ourensemble1/model1/best_segmentation_model1.pth'
MODEL2_PATH = '/kaggle/input/datasets/deblinamazumdersetu/ourensemble1/model1/best_transunetpp_model.pth'
MODEL3_PATH = '/kaggle/input/datasets/deblinamazumdersetu/model3d/best_segmentation_model3.pth'
METADATA_PATH = '/kaggle/input/datasets/deblinamazumdersetu/t2metadata/T2_age_gender_evaluation.csv'
TEST_DIR = '/kaggle/input/datasets/deblinamazumdersetu/datat2/Cirrhosis_T2_2D/test'
IMAGE_SIZE = (224, 224)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =================================================================================
# 2. UTILITY FUNCTIONS
# =================================================================================

def calculate_metrics(preds, targets, smooth=1e-6):
    """Calculates Dice and IoU from raw logits."""
    preds_sig = torch.sigmoid(preds)
    preds_bin = (preds_sig > 0.5).float()
    preds_flat, targets_flat = preds_bin.view(-1), targets.view(-1) 
    intersection = (preds_flat * targets_flat).sum()
    total = preds_flat.sum() + targets_flat.sum()
    union = total - intersection
    dice = (2. * intersection + smooth) / (total + smooth)
    iou = (intersection + smooth) / (union + smooth)
    return dice.item(), iou.item()

# =================================================================================
# 3. DATASET & TRANSFORM CLASSES
# =================================================================================

class SegmentationTransform:
    def __init__(self, image_size=IMAGE_SIZE):
        self.image_size = image_size
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    def __call__(self, image, mask):
        image = image.resize(self.image_size, Image.BILINEAR)
        mask = mask.resize(self.image_size, Image.NEAREST)
        image_tensor = transforms.ToTensor()(image)
        mask_tensor = (transforms.ToTensor()(mask) > 0).float()
        return self.normalize(image_tensor), mask_tensor

class LiverDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe
        self.transform = transform
    def __len__(self): return len(self.dataframe)
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_pil = Image.open(row['image_file_path']).convert('RGB')
        mask_pil = Image.open(row['mask_file_path']).convert('L')
        image, mask = self.transform(image_pil, mask_pil)
        return image, mask, row['class_name']

# =================================================================================
# 4. MODEL ARCHITECTURES
# =================================================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__(); self.conv = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.conv(x)

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__(); self.conv1 = nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(out_channels); self.relu = nn.ReLU(inplace=True); self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(out_channels)
    def forward(self, x, skip_connection):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True); x = torch.cat([x, skip_connection], dim=1)
        x = self.relu(self.bn1(self.conv1(x))); x = self.relu(self.bn2(self.conv2(x))); return x

class UNetWithResNet50Encoder(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__(); base_model = models.resnet50(weights=None); base_layers = list(base_model.children()); self.encoder0, self.encoder1 = nn.Sequential(*base_layers[:3]), nn.Sequential(*base_layers[3:5]); self.encoder2, self.encoder3, self.encoder4 = base_layers[5], base_layers[6], base_layers[7]; self.decoder3 = DecoderBlock(2048, 1024, 512); self.decoder2 = DecoderBlock(512, 512, 256); self.decoder1 = DecoderBlock(256, 256, 128); self.decoder0 = DecoderBlock(128, 64, 64); self.final_conv = nn.Conv2d(64, n_classes, kernel_size=1)
    def forward(self, x):
        e0=self.encoder0(x); e1=self.encoder1(e0); e2=self.encoder2(e1); e3=self.encoder3(e2); e4=self.encoder4(e3); d3=self.decoder3(e4,e3); d2=self.decoder2(d3,e2); d1=self.decoder1(d2,e1); d0=self.decoder0(d1,e0); out = F.interpolate(d0, scale_factor=2, mode='bilinear', align_corners=True); return self.final_conv(out)

class TransUNetPP(nn.Module):
    def __init__(self, n_classes=1, img_dim=224, vit_dim=768, vit_depth=12, vit_heads=12):
        super().__init__(); base_model = models.resnet50(weights=None); base_layers = list(base_model.children()); self.encoder0, self.encoder1 = nn.Sequential(*base_layers[:3]), nn.Sequential(*base_layers[3:5]); self.encoder2, self.encoder3, self.encoder4 = base_layers[5], base_layers[6], base_layers[7]; num_patches, self.patch_dim = (img_dim // 32) ** 2, 2048; self.pos_embedding = nn.Parameter(torch.randn(1, num_patches, vit_dim)); self.patch_to_embedding = nn.Linear(self.patch_dim, vit_dim); transformer_layer = nn.TransformerEncoderLayer(d_model=vit_dim, nhead=vit_heads, dim_feedforward=vit_dim * 4, batch_first=True); self.transformer_encoder = nn.TransformerEncoder(transformer_layer, num_layers=vit_depth); self.transformer_output_to_conv = nn.Sequential(nn.Linear(vit_dim, self.patch_dim), nn.LayerNorm(self.patch_dim)); d_ch = {'d0': 64, 'd1': 128, 'd2': 256, 'd3': 512, 'd4': 1024}; self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True); self.X_0_0=ConvBlock(64,d_ch['d0']); self.X_1_0=ConvBlock(256,d_ch['d1']); self.X_0_1=ConvBlock(d_ch['d0']+d_ch['d1'],d_ch['d0']); self.X_2_0=ConvBlock(512,d_ch['d2']); self.X_1_1=ConvBlock(d_ch['d1']+d_ch['d2'],d_ch['d1']); self.X_0_2=ConvBlock(d_ch['d0']*2+d_ch['d1'],d_ch['d0']); self.X_3_0=ConvBlock(1024,d_ch['d3']); self.X_2_1=ConvBlock(d_ch['d2']+d_ch['d3'],d_ch['d2']); self.X_1_2=ConvBlock(d_ch['d1']*2+d_ch['d2'],d_ch['d1']); self.X_0_3=ConvBlock(d_ch['d0']*3+d_ch['d1'],d_ch['d0']); self.X_4_0=ConvBlock(2048,d_ch['d4']); self.X_3_1=ConvBlock(d_ch['d3']+d_ch['d4'],d_ch['d3']); self.X_2_2=ConvBlock(d_ch['d2']*2+d_ch['d3'],d_ch['d2']); self.X_1_3=ConvBlock(d_ch['d1']*3+d_ch['d2'],d_ch['d1']); self.X_0_4=ConvBlock(d_ch['d0']*4+d_ch['d1'],d_ch['d0']); self.final_conv = nn.Conv2d(d_ch['d0'], n_classes, kernel_size=1)
    def forward(self, x):
        e0=self.encoder0(x); e1=self.encoder1(e0); e2=self.encoder2(e1); e3=self.encoder3(e2); e4=self.encoder4(e3); bs,_,h,w = e4.shape; trans_in = self.patch_to_embedding(e4.flatten(2).transpose(1, 2)) + self.pos_embedding; trans_out = self.transformer_output_to_conv(self.transformer_encoder(trans_in)).transpose(1, 2).view(bs, self.patch_dim, h, w); x0_0=self.X_0_0(e0); x1_0=self.X_1_0(e1); x0_1=self.X_0_1(torch.cat([x0_0,self.upsample(x1_0)],1)); x2_0=self.X_2_0(e2); x1_1=self.X_1_1(torch.cat([x1_0,self.upsample(x2_0)],1)); x0_2=self.X_0_2(torch.cat([x0_0,x0_1,self.upsample(x1_1)],1)); x3_0=self.X_3_0(e3); x2_1=self.X_2_1(torch.cat([x2_0,self.upsample(x3_0)],1)); x1_2=self.X_1_2(torch.cat([x1_0,x1_1,self.upsample(x2_1)],1)); x0_3=self.X_0_3(torch.cat([x0_0,x0_1,x0_2,self.upsample(x1_2)],1)); x4_0=self.X_4_0(trans_out); x3_1=self.X_3_1(torch.cat([x3_0,self.upsample(x4_0)],1)); x2_2=self.X_2_2(torch.cat([x2_0,x2_1,self.upsample(x3_1)],1)); x1_3=self.X_1_3(torch.cat([x1_0,x1_1,x1_2,self.upsample(x2_2)],1)); x0_4=self.X_0_4(torch.cat([x0_0,x0_1,x0_2,x0_3,self.upsample(x1_3)],1)); out = F.interpolate(self.final_conv(x0_4), scale_factor=2, mode='bilinear', align_corners=True); return out

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__(); self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int)); self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int)); self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid()); self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        psi = self.relu(self.W_g(g) + self.W_x(x)); return x * self.psi(psi)

class AttentionUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__(); base = models.resnet50(weights=None); layers = list(base.children()); self.encoder0, self.encoder1 = nn.Sequential(*layers[:3]), nn.Sequential(*layers[3:5]); self.encoder2, self.encoder3, self.encoder4 = layers[5], layers[6], layers[7]; self.upconv3 = nn.ConvTranspose2d(2048, 1024, 2, 2); self.attn3 = AttentionGate(1024, 1024, 512); self.dec_conv3 = ConvBlock(2048, 1024); self.upconv2 = nn.ConvTranspose2d(1024, 512, 2, 2); self.attn2 = AttentionGate(512, 512, 256); self.dec_conv2 = ConvBlock(1024, 512); self.upconv1 = nn.ConvTranspose2d(512, 256, 2, 2); self.attn1 = AttentionGate(256, 256, 128); self.dec_conv1 = ConvBlock(512, 256); self.upconv0 = nn.ConvTranspose2d(256, 64, 2, 2); self.attn0 = AttentionGate(64, 64, 32); self.dec_conv0 = ConvBlock(128, 64); self.final_up = nn.ConvTranspose2d(64, 32, 2, 2); self.final_conv = nn.Conv2d(32, n_classes, 1)
    def forward(self, x):
        e0=self.encoder0(x); e1=self.encoder1(e0); e2=self.encoder2(e1); e3=self.encoder3(e2); e4=self.encoder4(e3); d3=self.upconv3(e4); x3=self.attn3(d3,e3); d3=self.dec_conv3(torch.cat((x3,d3),1)); d2=self.upconv2(d3); x2=self.attn2(d2,e2); d2=self.dec_conv2(torch.cat((x2,d2),1)); d1=self.upconv1(d2); x1=self.attn1(d1,e1); d1=self.dec_conv1(torch.cat((x1,d1),1)); d0=self.upconv0(d1); x0=self.attn0(d0,e0); d0=self.dec_conv0(torch.cat((x0,d0),1)); return self.final_conv(self.final_up(d0))

# =================================================================================
# 5. EXECUTION: ABLATION STUDY
# =================================================================================

# --- 5.1 Load Data ---
metadata_df = pd.read_csv(METADATA_PATH)
metadata_df.rename(columns={'Patient ID': 'ID', 'Radiological Evaluation': 'radiological_evaluation'}, inplace=True)
metadata_df['ID'] = metadata_df['ID'].astype(str)
metadata_df['class_name'] = metadata_df['radiological_evaluation'].map({1: 'Mild', 2: 'Moderate', 3: 'Severe'})

def get_test_files(directory, allowed_ids):
    data = []
    for f in os.listdir(directory):
        if f in allowed_ids:
            im_path = os.path.join(directory, f, 'images')
            mk_path = os.path.join(directory, f, 'masks')
            for img in os.listdir(im_path):
                data.append((f, os.path.join(im_path, img), os.path.join(mk_path, img)))
    return pd.DataFrame(data, columns=['ID', 'image_file_path', 'mask_file_path'])

test_df = get_test_files(TEST_DIR, set(metadata_df['ID']))
test_df = pd.merge(test_df, metadata_df[['ID', 'class_name']], on='ID')
test_loader = DataLoader(LiverDataset(test_df, SegmentationTransform()), batch_size=16, shuffle=False)

# --- 5.2 Load Models ---
print("Loading Pre-trained Models...")
m1 = UNetWithResNet50Encoder().to(DEVICE); m1.load_state_dict(torch.load(MODEL1_PATH, map_location=DEVICE)); m1.eval()
m2 = TransUNetPP().to(DEVICE); m2.load_state_dict(torch.load(MODEL2_PATH, map_location=DEVICE)); m2.eval()
m3 = AttentionUNet().to(DEVICE); m3.load_state_dict(torch.load(MODEL3_PATH, map_location=DEVICE)); m3.eval()

# --- 5.3 Extract Predictions ---
print("Running Inference for Ablation Study...")
preds = {"M1": [], "M2": [], "M3": []}; gts = []; classes = []
with torch.no_grad():
    for images, masks, class_names in tqdm(test_loader):
        images = images.to(DEVICE)
        preds["M1"].append(m1(images).cpu())
        preds["M2"].append(m2(images).cpu())
        preds["M3"].append(m3(images).cpu())
        gts.append(masks.cpu()); classes.extend(class_names)

# Flatten tensors
p_map = {k: torch.cat(v) for k, v in preds.items()}; gt_tensor = torch.cat(gts)

# --- 5.4 Calculate Ablation Table ---
combinations = {
    "UNet-ResNet50 (M1)": ["M1"], "TransUNet++ (M2)": ["M2"], "Attention U-Net (M3)": ["M3"],
    "Dual: M1 + M2": ["M1", "M2"], "Dual: M1 + M3": ["M1", "M3"], "Dual: M2 + M3": ["M2", "M3"],
    "Proposed 3UResNet (M1+M2+M3)": ["M1", "M2", "M3"]
}

results = []
for name, keys in combinations.items():
    combined_logits = torch.mean(torch.stack([p_map[k] for k in keys]), dim=0)
    ov_dice, ov_iou = calculate_metrics(combined_logits, gt_tensor)
    
    # Class-wise metrics
    temp_df = pd.DataFrame({'logits': list(combined_logits.unbind(0)), 'gt': list(gt_tensor.unbind(0)), 'cls': classes})
    c_stats = {}
    for cn, group in temp_df.groupby('cls'):
        c_stats[cn], _ = calculate_metrics(torch.stack(group['logits'].tolist()), torch.stack(group['gt'].tolist()))
    
    results.append({"Configuration": name, "Overall Dice": ov_dice, "Overall mIoU": ov_iou, 
                    "Mild": c_stats.get('Mild', 0), "Moderate": c_stats.get('Moderate', 0), "Severe": c_stats.get('Severe', 0)})

# --- 5.5 Print Results ---
print("\n" + "="*95 + "\nFINAL ABLATION STUDY TABLE\n" + "="*95)
print(pd.DataFrame(results).to_string(index=False, float_format=lambda x: "{:.4f}".format(x)))
print("="*95)

Loading Pre-trained Models...
Running Inference for Ablation Study...


100%|██████████| 42/42 [16:17<00:00, 23.28s/it]



FINAL ABLATION STUDY TABLE
               Configuration  Overall Dice  Overall mIoU   Mild  Moderate  Severe
          UNet-ResNet50 (M1)        0.9483        0.9017 0.9493    0.9478  0.9461
            TransUNet++ (M2)        0.9416        0.8896 0.9434    0.9421  0.9359
        Attention U-Net (M3)        0.9477        0.9007 0.9483    0.9514  0.9427
               Dual: M1 + M2        0.9500        0.9048 0.9511    0.9479  0.9492
               Dual: M1 + M3        0.9510        0.9065 0.9520    0.9518  0.9473
               Dual: M2 + M3        0.9503        0.9053 0.9509    0.9515  0.9473
Proposed 3UResNet (M1+M2+M3)        0.9516        0.9077 0.9523    0.9524  0.9490
